In [11]:
import pandas as pd
import numpy as np


from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score



In [18]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("glowstudygram/spotify-songs-and-artists-dataset")

print("Path to dataset files:", path)

c:\Users\gaboe\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 66.8k/66.8k [00:00<00:00, 3.10MB/s]

Extracting files...
Path to dataset files: C:\Users\gaboe\.cache\kagglehub\datasets\glowstudygram\spotify-songs-and-artists-dataset\versions\1


In [26]:
ruta_archivo = "C:/Users/gaboe/Downloads/spotifydataset.csv"


try:
    df = pd.read_csv(ruta_archivo)
    print(f"Datos cargados correctamente: {df.shape[0]} filas y {df.shape[1]} columnas.")
    display(df.head())
except FileNotFoundError:
    raise FileNotFoundError(
        f"No se encontró '{ruta_archivo}'. Descarga el CSV del conjunto de datos "
        "de Kaggle y colócalo en la carpeta del notebook, o actualiza la ruta."
    )
except Exception as error:
    raise RuntimeError(f"No se pudo cargar el archivo: {error}")

Datos cargados correctamente: 1000 filas y 23 columnas.


,Unnamed: 0,artist_name,genres,followers,artist_popularity,artist_url,track_name,album_name,release_date,duration_ms,...,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,0,Ariana Grande,pop,98934105,89,https://open.spotify.com/artist/66CXWjxzNUsdJx...,we can't be friends (wait for your love),eternal sunshine,2024-03-08,228639,...,0.646,5,-8.334,1,0.0427,0.0615,0.000030,0.0740,0.295,115.842
1,1,Ariana Grande,pop,98934105,85,https://open.spotify.com/artist/66CXWjxzNUsdJx...,the boy is mine,eternal sunshine,2024-03-08,173639,...,0.630,7,-5.854,0,0.0434,0.1570,0.000000,0.0732,0.447,97.998
2,2,Ariana Grande,pop,98934105,83,https://open.spotify.com/artist/66CXWjxzNUsdJx...,intro (end of the world),eternal sunshine,2024-03-08,92400,...,0.362,10,-9.480,1,0.0416,0.6700,0.000000,0.1760,0.385,84.726
3,3,Ariana Grande,pop,98934105,80,https://open.spotify.com/artist/66CXWjxzNUsdJx...,Save Your Tears (Remix) (with Ariana Grande) -...,After Hours (Deluxe),2020-03-20,191013,...,0.825,0,-4.645,1,0.0325,0.0215,0.000024,0.0936,0.593,118.091
4,4,Ariana Grande,pop,98934105,79,https://open.spotify.com/artist/66CXWjxzNUsdJx...,"yes, and?",eternal sunshine,2024-03-08,214994,...,0.775,1,-6.614,1,0.0548,0.1900,0.000065,0.1130,0.787,118.998


In [27]:
columnas = [
    "track_popularity",
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
]

faltantes = [columna for columna in columnas if columna not in df.columns]
if faltantes:
    raise ValueError(
        "No se encontraron estas columnas en el CSV: "
        + ", ".join(faltantes)
        + f"\nColumnas disponibles: {list(df.columns)}"
    )

datos = df[columnas].copy()
print(f"Variables seleccionadas: {len(columnas)}")
display(datos.head())

Variables seleccionadas: 12


,track_popularity,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,89,0.645,0.646,5,-8.334,1,0.0427,0.0615,0.000030,0.0740,0.295,115.842
1,85,0.795,0.630,7,-5.854,0,0.0434,0.1570,0.000000,0.0732,0.447,97.998
2,83,0.506,0.362,10,-9.480,1,0.0416,0.6700,0.000000,0.1760,0.385,84.726
3,80,0.650,0.825,0,-4.645,1,0.0325,0.0215,0.000024,0.0936,0.593,118.091
4,79,0.787,0.775,1,-6.614,1,0.0548,0.1900,0.000065,0.1130,0.787,118.998


In [28]:
# Convertir valores a formato numérico y reemplazar valores no válidos con NaN.
for columna in columnas:
    datos[columna] = pd.to_numeric(datos[columna], errors="coerce")

# Reemplazar infinitos, eliminar filas completamente vacías y descartar duplicados.
datos = datos.replace([np.inf, -np.inf], np.nan)
datos = datos.dropna(how="all").drop_duplicates()

# Imputar valores faltantes con la mediana de cada variable.
faltantes_antes = int(datos.isna().sum().sum())
datos = datos.fillna(datos.median(numeric_only=True))

# Eliminar filas que sigan incompletas, por ejemplo, si una columna no tiene valores válidos.
datos = datos.dropna()

if datos.empty:
    raise ValueError("No quedaron filas válidas después de la limpieza.")

print(f"Filas disponibles después de la limpieza: {len(datos)}")
print(f"Valores faltantes imputados: {faltantes_antes}")
print("Valores faltantes restantes:", int(datos.isna().sum().sum()))
display(datos.describe().T)

Filas disponibles después de la limpieza: 993
Valores faltantes imputados: 0
Valores faltantes restantes: 0


,count,mean,std,min,25%,50%,75%,max
track_popularity,993.0,59.297080,23.913431,0.000000,57.0000,67.000000,73.00000,92.000
danceability,993.0,0.610932,0.167669,0.069000,0.5040,0.633000,0.72900,0.964
energy,993.0,0.659899,0.207749,0.003540,0.5380,0.680000,0.82000,0.998
key,993.0,5.243706,3.617139,0.000000,2.0000,5.000000,9.00000,11.000
loudness,993.0,-7.175998,4.489540,-39.482000,-8.3360,-5.948000,-4.58500,0.273
mode,993.0,0.613293,0.487241,0.000000,0.0000,1.000000,1.00000,1.000
speechiness,993.0,0.091254,0.090695,0.023200,0.0379,0.054500,0.10400,0.874
acousticness,993.0,0.265390,0.289730,0.000005,0.0281,0.151000,0.41700,0.996
instrumentalness,993.0,0.075216,0.226182,0.000000,0.0000,0.000004,0.00062,0.972
liveness,993.0,0.189109,0.145732,0.026000,0.0958,0.129000,0.24600,0.984


In [29]:
escalador = StandardScaler()
datos_estandarizados = escalador.fit_transform(datos)

print("Dimensiones de los datos estandarizados:", datos_estandarizados.shape)

Dimensiones de los datos estandarizados: (993, 12)


In [30]:

pca = PCA(n_components=2, random_state=42)
componentes = pca.fit_transform(datos_estandarizados)

pca_df = pd.DataFrame(
    componentes,
    columns=["Componente principal 1", "Componente principal 2"],
    index=datos.index,
)

varianza_explicada = pca.explained_variance_ratio_
print(f"Varianza explicada por el componente 1: {varianza_explicada[0]:.2%}")
print(f"Varianza explicada por el componente 2: {varianza_explicada[1]:.2%}")
print(f"Varianza explicada acumulada: {varianza_explicada.sum():.2%}")


Varianza explicada por el componente 1: 24.92%
Varianza explicada por el componente 2: 12.74%
Varianza explicada acumulada: 37.65%


In [ ]:
# Se prueban varios números de grupos y se elige el de mayor puntuación de silueta.
if len(datos) < 3:
    raise ValueError("Se necesitan al menos 3 canciones para evaluar el clustering.")

resultados = []
max_k = min(8, len(datos) - 1)

for k in range(2, max_k + 1):
    modelo = KMeans(n_clusters=k, random_state=42, n_init=10)
    etiquetas = modelo.fit_predict(datos_estandarizados)

    if len(set(etiquetas)) > 1:
        puntuacion = silhouette_score(datos_estandarizados, etiquetas)
        resultados.append((k, puntuacion))

if not resultados:
    raise ValueError("No fue posible calcular una puntuación de silueta.")

mejor_k, mejor_silueta = max(resultados, key=lambda resultado: resultado[1])
kmeans = KMeans(n_clusters=mejor_k, random_state=42, n_init=10)
etiquetas = kmeans.fit_predict(datos_estandarizados)

pca_df["Grupo"] = etiquetas.astype(str)
datos_con_grupo = datos.copy()
datos_con_grupo["Grupo"] = etiquetas

print(f"Número de grupos seleccionado: {mejor_k}")
print(f"Puntuación de silueta: {mejor_silueta:.3f}")

# el for in range me lo enseñaron en la universidad :)

Número de grupos seleccionado: 2
Puntuación de silueta: 0.273
